In [21]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, GlobalAveragePooling2D, 
    Input, LayerNormalization, Conv2D, BatchNormalization
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
from tqdm import tqdm

In [22]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [23]:
class VideoClassifier:
    
    def __init__(self, max_frames=30, frame_size=(224, 224)):
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.feature_extractor = None
        self.model = None
        logger.info(f"Initialized VideoClassifier with max_frames={max_frames}, frame_size={frame_size}")
        
    def _build_feature_extractor(self):
        
        # Create base ResNet model
        base_model = ResNet50V2(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.frame_size, 3)
        )
        
        # Unfreeze the last few layers for fine-tuning
        for layer in base_model.layers[:-30]:  # Freeze all except last 30 layers
            layer.trainable = False
            
        # Add custom layers on top
        
        x = base_model.output
        x = Conv2D(512, (1, 1), activation='relu')(x)
        x = BatchNormalization()(x)
        x = GlobalAveragePooling2D()(x)
        
        model = Model(inputs=base_model.input, outputs=x)
        return model 
    
    def extract_frames(self, video_path):
        try:
            if not str(video_path).endswith('.mp4'):
                video_path = str(video_path) + '.mp4'
            
            if not os.path.exists(video_path):
                logger.warning(f"Video file not found: {video_path}")
                return None
                
            frames = []
            cap = cv2.VideoCapture(video_path)
            
            if not cap.isOpened():
                logger.warning(f"Failed to open video: {video_path}")
                return None

            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            sample_interval = max(1, total_frames // self.max_frames)
            frame_count = 0

            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_count % sample_interval == 0:
                    frame = cv2.resize(frame, self.frame_size)
                    frames.append(frame)

                frame_count += 1

            cap.release()

            # Pad with zeros if needed
            while len(frames) < self.max_frames:
                frames.append(np.zeros((*self.frame_size, 3), dtype=np.uint8))

            return np.array(frames[:self.max_frames])
                
        except Exception as e:
            logger.error(f"Error processing video {video_path}: {str(e)}")
            if 'cap' in locals():
                cap.release()
        return None

    def prepare_data(self, video_dir, metadata_path, batch_size=32):
        
        try:
            video_dir = Path(video_dir)
            metadata_path = Path(metadata_path)
            
            if not video_dir.exists():
                raise ValueError(f"Video directory not found: {video_dir}")
            if not metadata_path.exists():
                raise ValueError(f"Metadata file not found: {metadata_path}")
                
            logger.info(f"Reading metadata from: {metadata_path}")
            metadata = pd.read_excel(metadata_path)
            logger.info(f"Original metadata shape: {metadata.shape}")
            
            if 'VideoID' not in metadata.columns or 'HowFunny' not in metadata.columns:
                raise ValueError("Required columns 'VideoID' or 'HowFunny' not found in metadata")
            
            logger.info("\nOriginal HowFunny values:\n%s", metadata['HowFunny'].value_counts())
            
            def combine_categories(x):
                if pd.isna(x):
                    return None
                x = str(x).strip()
                if x in ['Funny', 'Very funny', 'Extremely funny']:
                    return 'Very Funny'
                elif x == 'Slightly funny':
                    return 'Slightly Funny'
                elif x == 'Not funny':
                    return 'Not Funny'
                else:
                    return None

            metadata['HowFunny_Combined'] = metadata['HowFunny'].apply(combine_categories)
            metadata = metadata.dropna(subset=['HowFunny_Combined']) 
            category_mapping = {
                
                'Not Funny': 0,
                'Slightly Funny': 1,
                'Very Funny': 2
            }
            metadata['Label'] = metadata['HowFunny_Combined'].map(category_mapping)
            
            logger.info("\nCombined category distribution:\n%s", metadata['HowFunny_Combined'].value_counts())
            logger.info("\nNumerical label distribution:\n%s", metadata['Label'].value_counts())
            
            self.feature_extractor = self._build_feature_extractor()
            
            videos_list = []
            labels = []
            processed_videos = []
            
            logger.info(f"\nProcessing {len(metadata)} videos...")
            
            # Process videos in batches for memory efficiency
            batch_frames = []
            batch_labels = []
            batch_ids = []
            
            for idx, row in tqdm(metadata.iterrows(), total=len(metadata)):
                try:
                    video_id = str(row['VideoID']).strip()
                    video_path = video_dir / video_id
                    
                    frames = self.extract_frames(video_path)
                    if frames is not None:
                        batch_frames.append(frames)
                        batch_labels.append(int(row['Label']))
                        batch_ids.append(video_id)
                        
                        # Process batch when it reaches batch_size
                        if len(batch_frames) == batch_size:
                            batch_features = self._process_batch(batch_frames)
                            videos_list.extend(batch_features)
                            labels.extend(batch_labels)
                            processed_videos.extend(batch_ids)
                            
                            # Clear batch
                            batch_frames = []
                            batch_labels = []
                            batch_ids = []
                            
                            logger.info(f"Successfully processed {len(videos_list)} videos")
                            
                except Exception as e:
                    logger.error(f"Error processing video {row['VideoID']}: {str(e)}")
                    continue
            
            # Process remaining videos in the last batch
            
            if batch_frames:
                
                batch_features = self._process_batch(batch_frames)
                videos_list.extend(batch_features)
                labels.extend(batch_labels)
                processed_videos.extend(batch_ids)
            
            if not videos_list:
                
                logger.error("\nNo videos were successfully processed.")
                raise ValueError("No valid videos processed")
                
            X = np.array(videos_list, dtype=np.float32)
            y = np.array(labels, dtype=np.int32)
            
            logger.info(f"\nFinal dataset:")
            logger.info(f"Features shape: {X.shape}")
            logger.info(f"Labels shape: {y.shape}")
            logger.info(f"Label distribution: {np.unique(y, return_counts=True)}")
            
            output_dir = Path('outputs')
            output_dir.mkdir(exist_ok=True)
            
            with open(output_dir / 'processed_videos.txt', 'w') as f:
                for vid in processed_videos:
                    f.write(f"{vid}\n")
            
            return X, y
            
        except Exception as e:
            logger.error(f"\nError preparing dataset: {str(e)}")
            import traceback
            logger.error(traceback.format_exc())
            raise
        
    def _process_batch(self, batch_frames):
        """Process a batch of frames through the feature extractor"""
        batch_features = []
        
        # Reshape batch for processing
        frames = np.vstack(batch_frames)
        
        # Extract features in smaller sub-batches to prevent OOM
        sub_batch_size = 32
        for i in range(0, len(frames), sub_batch_size):
            sub_batch = frames[i:i + sub_batch_size]
            features = self.feature_extractor.predict(sub_batch, verbose=0)
            batch_features.extend(features)
        
        # Reshape features back to video sequences
        num_videos = len(batch_frames)
        frames_per_video = len(batch_frames[0])
        features_dim = batch_features[0].shape[0]
        
        return np.array(batch_features).reshape(num_videos, frames_per_video, features_dim)
    def build_model(self, input_shape, num_classes):
        
        inputs = Input(shape=input_shape)
    
        # Deeper LSTM architecture for fine-tuned features
        x = LSTM(512, return_sequences=True)(inputs)
        x = LayerNormalization()(x)
        x = Dropout(0.2)(x)
        
        x = LSTM(256, return_sequences=True)(x)
        x = LayerNormalization()(x)
        x = Dropout(0.2)(x)
        
        x = LSTM(128)(x)
        x = LayerNormalization()(x)
        x = Dropout(0.2)(x)
        
        x = Dense(256, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.2)(x)
        
        x = Dense(128, activation='relu')(x)
        x = BatchNormalization()(x)
        x = Dropout(0.1)(x)
        
        outputs = Dense(num_classes, activation='softmax')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        
        optimizer = Adam(learning_rate=1e-4)
        
        model.compile(
            optimizer=optimizer,
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model
    
    def train(self, video_dir, metadata_path, epochs=50, batch_size=32):
        X, y = self.prepare_data(video_dir, metadata_path, batch_size=batch_size)
        
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )
        
        input_shape = (X.shape[1], X.shape[2])
        
        self.model = self.build_model(
            input_shape=input_shape,
            num_classes=len(np.unique(y))
        )
        
        output_dir = Path('unimodal/cnnLstm_pretrain/normal/outputs')
        output_dir.mkdir(exist_ok=True)
        
        # Enhanced callbacks with ReduceLROnPlateau for dynamic learning rate adjustment
        
        callbacks = [
            EarlyStopping(
                monitor='val_accuracy',
                patience=15,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                output_dir / 'best_model.keras',
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.2,
                patience=5,
                min_lr=1e-7,
                verbose=1
            )
        ]
        
        # Train with class weights
        class_weights = dict(enumerate(
            len(y_train) / (len(np.unique(y_train)) * np.bincount(y_train))
        ))
        
        history = self.model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            class_weight=class_weights,
            verbose=1
        )
        
        self.evaluate(X_test, y_test, history)
        
    def evaluate(self, X_test, y_test, history):
        output_dir = Path('unimodal/cnnLstm_pretrain/normal/outputs')
        output_dir.mkdir(exist_ok=True)
        
        # Get predictions
        y_pred = np.argmax(self.model.predict(X_test), axis=1)
        
        # Plot training history
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 2, 1)
        plt.plot(history.history['accuracy'], label='Train')
        plt.plot(history.history['val_accuracy'], label='Validation')
        plt.title('Model Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend()
        
        plt.subplot(1, 2, 2)
        plt.plot(history.history['loss'], label='Train')
        plt.plot(history.history['val_loss'], label='Validation')
        plt.title('Model Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig(output_dir / 'training_history.png')
        plt.close()
        
        # Enhanced confusion matrix
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            xticklabels=['Not Funny', 'Slightly Funny', 'Very Funny'],
            yticklabels=['Not Funny', 'Slightly Funny', 'Very Funny']
        )
        plt.title('Confusion Matrix')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.savefig(output_dir / 'confusion_matrix.png', bbox_inches='tight')
        plt.close()
        
        # Save detailed classification report
        report = classification_report(
            y_test,
            y_pred,
            target_names=['Not Funny', 'Slightly Funny', 'Very Funny'],
            digits=4
        )
        logger.info("\nClassification Report:\n%s", report)
        
        with open(output_dir / 'classification_report.txt', 'w') as f:
            f.write(report)
        
        # Save model architecture
        with open(output_dir / 'model_summary.txt', 'w') as f:
            self.model.summary(print_fn=lambda x: f.write(x + '\n'))
    

In [24]:
try:
    video_dir = '/Users/jeremiah/phd_thesis/data/main_data/videos'
    metadata_path = 'data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx'
    
    # Setting random seeds for reproducibility
    np.random.seed(42)
    tf.random.set_seed(42)
    
    # Configure GPU memory growth
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    
    logger.info(f"Video directory: {video_dir}")
    logger.info(f"Metadata path: {metadata_path}")
    
    classifier = VideoClassifier(max_frames=20, frame_size=(224, 224))
    classifier.train(
        video_dir=video_dir,
        metadata_path=metadata_path,
        epochs=50,
        batch_size=16
    )
except Exception as e:
    logger.error(f"Error in main: {str(e)}")
    import traceback
    logger.error(traceback.format_exc())


2025-01-04 20:57:12,255 - INFO - Video directory: /Users/jeremiah/phd_thesis/data/main_data/videos
2025-01-04 20:57:12,256 - INFO - Metadata path: data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx
2025-01-04 20:57:12,257 - INFO - Initialized VideoClassifier with max_frames=20, frame_size=(224, 224)
2025-01-04 20:57:12,259 - INFO - Reading metadata from: data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx
2025-01-04 20:57:12,444 - INFO - Original metadata shape: (827, 25)
2025-01-04 20:57:12,448 - INFO - 
Original HowFunny values:
HowFunny
Not funny          318
Slightly funny     315
Funny              145
Very funny          44
Extremely funny      4
Name: count, dtype: int64
2025-01-04 20:57:12,456 - INFO - 
Combined category distribution:
HowFunny_Combined
Not Funny         318
Slightly Funny    315
Very Funny        193
Name: count, dtype: int64
2025-01-04 20:57:12,457 - INFO - 
Numerical label distribution:
Label
0    318
1    315
2    193
Name:

Epoch 1/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.3451 - loss: 1.6634
Epoch 1: val_accuracy improved from -inf to 0.37121, saving model to unimodal/cnnLstm_pretrain/normal/outputs/best_model.keras
33/33 ━━━━━━━━━━━━━━━━━━━━ 5s 84ms/step - accuracy: 0.3447 - loss: 1.6631 - val_accuracy: 0.3712 - val_loss: 1.0887 - learning_rate: 1.0000e-04
Epoch 2/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.3479 - loss: 1.5805
Epoch 2: val_accuracy improved from 0.37121 to 0.45455, saving model to unimodal/cnnLstm_pretrain/normal/outputs/best_model.keras
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.3475 - loss: 1.5813 - val_accuracy: 0.4545 - val_loss: 1.0684 - learning_rate: 1.0000e-04
Epoch 3/50
33/33 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.2813 - loss: 1.7177
Epoch 3: val_accuracy did not improve from 0.45455
33/33 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - accuracy: 0.2819 - loss: 1.7160 - val_accuracy: 0.2955 - val_loss: 1.1401 - learning_rate: 1.0000e-04
Epoch

2025-01-04 21:05:40,761 - INFO - 
Classification Report:
                precision    recall  f1-score   support

     Not Funny     0.3670    0.6250    0.4624        64
Slightly Funny     0.3878    0.3016    0.3393        63
    Very Funny     0.1250    0.0256    0.0426        39

      accuracy                         0.3614       166
     macro avg     0.2932    0.3174    0.2814       166
  weighted avg     0.3180    0.3614    0.3170       166



# With Data Augumentation

In [25]:
import os
import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.applications.resnet_v2 import preprocess_input
from tensorflow.keras.layers import (
    LSTM, Dense, Dropout, GlobalAveragePooling2D, 
    Input, LayerNormalization, Conv2D, BatchNormalization
)
from tensorflow.keras.models import Model
#import tensorflow_addons as tfa
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
from tqdm import tqdm
import random

In [26]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [27]:
class VideoAugmenter:
    @staticmethod
    def apply_augmentation(frame):
        """Apply various augmentation techniques to a single frame"""
        try:
            # Convert to tensor
            frame = tf.convert_to_tensor(frame)
            frame = tf.expand_dims(frame, 0)
            
            # Random horizontal flip
            if random.random() > 0.5:
                frame = tf.image.random_flip_left_right(frame)
            
            # Random rotation
            if random.random() > 0.5:
                angle = random.uniform(-15, 15)
                frame = tfa.image.rotate(frame, angle * np.pi / 180)
            
            # Random brightness
            if random.random() > 0.5:
                frame = tf.image.random_brightness(frame, 0.2)
            
            # Random contrast
            if random.random() > 0.5:
                frame = tf.image.random_contrast(frame, 0.8, 1.2)
            
            # Random saturation
            if random.random() > 0.5:
                frame = tf.image.random_saturation(frame, 0.8, 1.2)
            
            # Random hue
            if random.random() > 0.5:
                frame = tf.image.random_hue(frame, 0.1)
            
            # Random zoom
            if random.random() > 0.5:
                scale = random.uniform(0.8, 1.2)
                frame = tf.image.resize(
                    frame,
                    tf.cast(tf.shape(frame)[1:3] * scale, tf.int32),
                    method=tf.image.ResizeMethod.BILINEAR
                )
                frame = tf.image.resize_with_crop_or_pad(
                    frame,
                    tf.shape(frame)[1],
                    tf.shape(frame)[2]
                )
            
            # Ensure values are in valid range
            frame = tf.clip_by_value(frame, 0, 1)
            frame = frame[0].numpy()
            
            return frame
            
        except Exception as e:
            logger.error(f"Error in frame augmentation: {str(e)}")
            return frame
    
    @staticmethod
    def temporal_augmentation(frames, max_frames):
        """Apply temporal augmentation to frame sequence"""
        try:
            num_frames = len(frames)
            if num_frames <= max_frames:
                return frames
            
            # Randomly sample frames while keeping first and last
            indices = sorted(random.sample(range(1, num_frames-1), max_frames-2))
            indices = [0] + indices + [num_frames-1]
            
            return frames[indices]
            
        except Exception as e:
            logger.error(f"Error in temporal augmentation: {str(e)}")
            return frames[:max_frames] 

In [28]:
class VideoClassifier: 
    
    def __init__(self, max_frames=30, frame_size=(224, 224)):
        self.max_frames = max_frames
        self.frame_size = frame_size
        self.feature_extractor = None
        self.model = None
        self.augmenter = VideoAugmenter()
        logger.info(f"Initialized VideoClassifier with max_frames={max_frames}, frame_size={frame_size}")
        
    def _build_feature_extractor(self):
        
        # Create base ResNet model
        base_model = ResNet50V2(
            weights='imagenet',
            include_top=False,
            input_shape=(*self.frame_size, 3)
        )
        
        # Unfreeze the last few layers for fine-tuning
        for layer in base_model.layers[:-30]:
            layer.trainable = False
            
        # Add custom layers on top
        x = base_model.output
        x = Conv2D(512, (1, 1), activation='relu')(x)
        x = BatchNormalization()(x)
        x = GlobalAveragePooling2D()(x)
        
        model = Model(inputs=base_model.input, outputs=x)
        return model
    
    def extract_frames(self, video_path, augment=False):
        
        """Extract frames from video with optional augmentation"""
        try:
            if not str(video_path).endswith('.mp4'):
                video_path = str(video_path) + '.mp4'
            
            if not os.path.exists(video_path):
                logger.warning(f"Video file not found: {video_path}")
                return None
                
            frames = []
            cap = cv2.VideoCapture(video_path)
            
            if not cap.isOpened():
                logger.warning(f"Failed to open video: {video_path}")
                return None

            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total_frames == 0:
                logger.warning(f"Video has no frames: {video_path}")
                return None

            # Extract all frames first
            all_frames = []
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.resize(frame, self.frame_size)
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = frame.astype(np.float32) / 255.0
                all_frames.append(frame)
            
            cap.release()
            
            if not all_frames:
                logger.warning(f"No frames extracted from video: {video_path}")
                return None
            
            # Apply temporal augmentation
            if augment:
                all_frames = self.augmenter.temporal_augmentation(all_frames, self.max_frames)
            else:
                # Regular frame sampling
                indices = np.linspace(0, len(all_frames) - 1, self.max_frames, dtype=int)
                all_frames = [all_frames[i] for i in indices]
            
            # Apply spatial augmentation if requested
            frames = []
            for frame in all_frames:
                if augment:
                    frame = self.augmenter.apply_augmentation(frame)
                frames.append(frame)
            
            if len(frames) < self.max_frames:
                logger.warning(f"Not enough frames in video: {video_path}")
                return None
                
            return np.array(frames)
            
        except Exception as e:
            logger.error(f"Error processing video {video_path}: {str(e)}")
            if 'cap' in locals():
                cap.release()
            return None 
        
    def prepare_data(self, video_dir, metadata_path, batch_size=32):
            try:
                video_dir = Path(video_dir)
                metadata_path = Path(metadata_path)
                
                if not video_dir.exists():
                    raise ValueError(f"Video directory not found: {video_dir}")
                if not metadata_path.exists():
                    raise ValueError(f"Metadata file not found: {metadata_path}")
                    
                logger.info(f"Reading metadata from: {metadata_path}")
                metadata = pd.read_excel(metadata_path)
                logger.info(f"Original metadata shape: {metadata.shape}")
                
                if 'VideoID' not in metadata.columns or 'HowFunny' not in metadata.columns:
                    raise ValueError("Required columns 'VideoID' or 'HowFunny' not found in metadata")
                
                logger.info("\nOriginal HowFunny values:\n%s", metadata['HowFunny'].value_counts())
                
                def combine_categories(x):
                    if pd.isna(x):
                        return None
                    x = str(x).strip()
                    if x in ['Funny', 'Very funny', 'Extremely funny']:
                        return 'Very Funny'
                    elif x == 'Slightly funny':
                        return 'Slightly Funny'
                    elif x == 'Not funny':
                        return 'Not Funny'
                    else:
                        return None
    
                metadata['HowFunny_Combined'] = metadata['HowFunny'].apply(combine_categories)
                metadata = metadata.dropna(subset=['HowFunny_Combined'])
                
                category_mapping = {
                    'Not Funny': 0,
                    'Slightly Funny': 1,
                    'Very Funny': 2
                }
                metadata['Label'] = metadata['HowFunny_Combined'].map(category_mapping)
                
                logger.info("\nCombined category distribution:\n%s", metadata['HowFunny_Combined'].value_counts())
                logger.info("\nNumerical label distribution:\n%s", metadata['Label'].value_counts())
                
                # Initialize feature extractor
                self.feature_extractor = self._build_feature_extractor()
                
                videos_list = []
                labels = []
                processed_videos = []
                
                logger.info(f"\nProcessing {len(metadata)} videos...")
                
                # Process videos in batches
                batch_frames = []
                batch_labels = []
                batch_ids = []
                
                for idx, row in tqdm(metadata.iterrows(), total=len(metadata)):
                    try:
                        video_id = str(row['VideoID']).strip()
                        video_path = video_dir / video_id
                        
                        # Process original frames
                        frames = self.extract_frames(video_path, augment=False)
                        if frames is not None:
                            batch_frames.append(frames)
                            batch_labels.append(int(row['Label']))
                            batch_ids.append(video_id)
                            

                            aug_frames = self.extract_frames(video_path, augment=True)
                            if aug_frames is not None:
                                batch_frames.append(aug_frames)
                                batch_labels.append(int(row['Label']))
                                batch_ids.append(video_id + "_augmented")
                            
                            if len(batch_frames) >= batch_size:
                                batch_features = self._process_batch(batch_frames)
                                videos_list.extend(batch_features)
                                labels.extend(batch_labels)
                                processed_videos.extend(batch_ids)
                                
                                batch_frames = []
                                batch_labels = []
                                batch_ids = []
                                
                                logger.info(f"Successfully processed {len(videos_list)} videos")
                                
                    except Exception as e:
                        logger.error(f"Error processing video {row['VideoID']}: {str(e)}")
                        continue
                    # Process remaining videos in the last batch
                    if batch_frames:
                        batch_features = self._process_batch(batch_frames)
                        videos_list.extend(batch_features)
                        labels.extend(batch_labels)
                        processed_videos.extend(batch_ids)
                    
                    if not videos_list:
                        logger.error("\nNo videos were successfully processed.")
                        raise ValueError("No valid videos processed")
                        
                    X = np.array(videos_list, dtype=np.float32)
                    y = np.array(labels, dtype=np.int32)
                    
                    logger.info(f"\nFinal dataset:")
                    logger.info(f"Features shape: {X.shape}")
                    logger.info(f"Labels shape: {y.shape}")
                    logger.info(f"Label distribution: {np.unique(y, return_counts=True)}")
                    
                    output_dir = Path('outputs')
                    output_dir.mkdir(exist_ok=True)
                    
                    with open(output_dir / 'processed_videos.txt', 'w') as f:
                        for vid in processed_videos:
                            f.write(f"{vid}\n")
                    
                    return X, y
                    
            except Exception as e:
                logger.error(f"\nError preparing dataset: {str(e)}")
                import traceback
                logger.error(traceback.format_exc())
                raise
            
    def _process_batch(self, batch_frames):
        
        """Process a batch of frames through the feature extractor"""
        batch_features = []
        
        frames = np.vstack(batch_frames)
        
        # Extract features in smaller sub-batches to prevent OOM
        sub_batch_size = 32
        for i in range(0, len(frames), sub_batch_size):
            sub_batch = frames[i:i + sub_batch_size]
            features = self.feature_extractor.predict(sub_batch, verbose=0)
            batch_features.extend(features)
        
        # Reshape features back to video sequences
        num_videos = len(batch_frames)
        frames_per_video = len(batch_frames[0])
        features_dim = batch_features[0].shape[0]
        
        return np.array(batch_features).reshape(num_videos, frames_per_video, features_dim)
    
    def build_model(self, input_shape, num_classes):
        
        inputs = Input(shape=input_shape)
        
        # Deeper LSTM architecture
        x = LSTM(512, return_sequences=True)(inputs)
        x = LayerNormalization()(x)
        x = Dropout(0.3)(x)
        
        x = LSTM(256, return_sequences=True)(x)
        x = LayerNormalization()(x)
        x = Dropout(0.3)(x)
        
        x = LSTM(128)(x)
        x = LayerNormalization()(x)
        x = Dropout(0.2)(x)
        
        # Dense layers with residual connections
        x1 = Dense(256, activation='relu')(x)
        x1 = BatchNormalization()(x1)
        x1 = Dropout(0.2)(x1)
        
        x2 = Dense(128, activation='relu')(x1)
        x2 = BatchNormalization()(x2)
        x2 = Dropout(0.1)(x2)
        
        # Concatenate features
        x = tf.keras.layers.Concatenate()([x1, x2])
        
        x = Dense(64, activation='relu')(x)
        outputs = Dense(num_classes, activation='softmax')(x)
        
        model = Model(inputs=inputs, outputs=outputs)
        
        optimizer = Adam(learning_rate=1e-4)
        
        model.compile(
            optimizer=optimizer,
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
        return model
    
    def train(self, video_dir, metadata_path, epochs=50, batch_size=32):
        X, y = self.prepare_data(video_dir, metadata_path, batch_size=batch_size)
        
        # Shuffle and split the data
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=442, stratify=y
        )
        
        input_shape = (X.shape[1], X.shape[2])
        
        self.model = self.build_model(
            input_shape=input_shape,
            num_classes=len(np.unique(y))
        )
        
        output_dir = Path('unimodal/cnnLstm_pretrain/aug/outputs')
        output_dir.mkdir(exist_ok=True)
        # Enhanced callbacks
        callbacks = [
            EarlyStopping(
                monitor='val_accuracy',
                patience=15,
                restore_best_weights=True,
                verbose=1
            ),
            ModelCheckpoint(
                output_dir / 'best_model.keras',
                monitor='val_accuracy',
                save_best_only=True,
                verbose=1
            ),
            ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.2,
                patience=5,
                min_lr=1e-7,
                verbose=1
            )
        ]
        
        # Calculate class weights for imbalanced dataset
        class_weights = dict(enumerate(
            len(y_train) / (len(np.unique(y_train)) * np.bincount(y_train))
        ))
        
        history = self.model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=callbacks,
            class_weight=class_weights,
            verbose=1
        )
        
        self.evaluate(X_test, y_test, history)
        
    def evaluate(self, X_test, y_test, history):
        output_dir = Path('unimodal/cnnLstm_pretrain/aug/outputs')
        output_dir.mkdir(exist_ok=True)
        
        # Get predictions
        y_pred = np.argmax(self.model.predict(X_test), axis=1)
        
        # Plot detailed training history
        metrics = ['accuracy', 'loss']
        fig, axes = plt.subplots(1, 2, figsize=(15, 5))
        
        for idx, metric in enumerate(metrics):
            axes[idx].plot(history.history[metric], label='Train')
            axes[idx].plot(history.history[f'val_{metric}'], label='Validation')
            axes[idx].set_title(f'Model {metric.capitalize()}')
            axes[idx].set_xlabel('Epoch')
            axes[idx].set_ylabel(metric.capitalize())
            axes[idx].legend()
            axes[idx].grid(True)
        
        plt.tight_layout()
        plt.savefig(output_dir / 'training_history.png')
        plt.close()
        
        # Enhanced confusion matrix
        plt.figure(figsize=(10, 8))
        cm = confusion_matrix(y_test, y_pred)
        cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        sns.heatmap(
            cm_normalized,
            annot=cm,
            fmt='d',
            cmap='Blues',
            xticklabels=['Not Funny', 'Slightly Funny', 'Very Funny'],
            yticklabels=['Not Funny', 'Slightly Funny', 'Very Funny']
        )
        plt.title('Confusion Matrix\n(values: counts, colors: percentages)')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.savefig(output_dir / 'confusion_matrix.png', bbox_inches='tight')
        plt.close()
        
        # Detailed classification report
        report = classification_report(
            y_test,
            y_pred,
            target_names=['Not Funny', 'Slightly Funny', 'Very Funny'],
            digits=4
        )
        logger.info("\nClassification Report:\n%s", report)
        
        # Save all evaluation results
        with open(output_dir / 'evaluation_results.txt', 'w') as f:
            f.write("Classification Report:\n")
            f.write(report)
            f.write("\n\nConfusion Matrix:\n")
            f.write(str(cm))
            f.write("\n\nNormalized Confusion Matrix:\n")
            f.write(str(cm_normalized))
        
        # Save model architecture
        with open(output_dir / 'model_summary.txt', 'w') as f:
            self.model.summary(print_fn=lambda x: f.write(x + '\n'))


In [29]:

    try:
        video_dir = '/Users/jeremiah/video-p-reaction-joketype-modular/data/videos'
        metadata_path = 'data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx'
        
        # Set random seeds for reproducibility
        np.random.seed(42)
        tf.random.set_seed(42)
        random.seed(42)
        
        gpus = tf.config.experimental.list_physical_devices('GPU')
        if gpus:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            logger.info(f"Found {len(gpus)} GPU(s)")
        else:
            logger.info("No GPUs found, using CPU")
        
        logger.info(f"Video directory: {video_dir}")
        logger.info(f"Metadata path: {metadata_path}")
        
        # Initialize and train the classifier
        classifier = VideoClassifier(max_frames=20, frame_size=(224, 224))
        classifier.train(
            video_dir=video_dir,
            metadata_path=metadata_path,
            epochs=50,
            batch_size=32
        )
        
    except Exception as e:
        logger.error(f"Error in main: {str(e)}")
        import traceback
        logger.error(traceback.format_exc()) 

2025-01-04 21:05:40,988 - INFO - No GPUs found, using CPU
2025-01-04 21:05:40,989 - INFO - Video directory: /Users/jeremiah/video-p-reaction-joketype-modular/data/videos
2025-01-04 21:05:40,989 - INFO - Metadata path: data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx
2025-01-04 21:05:40,990 - INFO - Initialized VideoClassifier with max_frames=20, frame_size=(224, 224)
2025-01-04 21:05:40,990 - INFO - Reading metadata from: data/main_data/excel2_modified_filtered_sorted_by_Total Seconds.xlsx
2025-01-04 21:05:41,105 - INFO - Original metadata shape: (827, 25)
2025-01-04 21:05:41,106 - INFO - 
Original HowFunny values:
HowFunny
Not funny          318
Slightly funny     315
Funny              145
Very funny          44
Extremely funny      4
Name: count, dtype: int64
2025-01-04 21:05:41,108 - INFO - 
Combined category distribution:
HowFunny_Combined
Not Funny         318
Slightly Funny    315
Very Funny        193
Name: count, dtype: int64
2025-01-04 21:05:41,109 - INFO 